In [1]:
# Import necessary packages

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

In [2]:
# Read in all CSV files from f1_data folder

constructors = pd.read_csv('f1_data/constructors.csv')
constructor_standings = pd.read_csv('f1_data/constructor_standings.csv')
constructor_results = pd.read_csv('f1_data/constructor_results.csv')
driver_standings = pd.read_csv('f1_data/driver_standings.csv')
drivers = pd.read_csv('f1_data/drivers.csv')
lap_times = pd.read_csv('f1_data/lap_times.csv')
pit_stops = pd.read_csv('f1_data/pit_stops.csv')
results = pd.read_csv('f1_data/results.csv')
races = pd.read_csv('f1_data/races.csv')


In [3]:
# Add years column to driver_standings df by merging with races df on raceId
driver_standings_w_years = pd.merge(driver_standings, races[['raceId', 'year']], on='raceId')
# Group driver_standings df by driverId and year, calculate mean points for each driver/year pair
driver_standings_summary = driver_standings_w_years.groupby(['driverId', 'year'], as_index=False).agg(mean_points = ('points', 'mean'))

# Add years to constructor_standings df by merging with races df on raceId
constructor_standings_w_years = pd.merge(constructor_standings, races[['raceId', 'year']], on='raceId')
# Group constructor_standings df by constructorId and year, calculate mean points for each constructor/year pair
constructor_standings_summary = constructor_standings_w_years.groupby(['constructorId', 'year'], as_index=False).agg(mean_points=('points', 'mean'))

In [4]:
# Add years column to pit_stops df by merging with races df on raceId
pit_stops_w_years = pd.merge(pit_stops, races[['raceId', 'year']], on='raceId')
# Add constructorId to pit_stops df by merging with results on raceId and driverId
pit_stops_w_constructors = pd.merge(pit_stops_w_years, results[['raceId', 'driverId', 'constructorId']], on=['raceId', 'driverId'])
# Convert milliseconds column to seconds
pit_stops_w_constructors['duration'] = pit_stops_w_constructors['milliseconds'] * 10**-3

# Group pit_stops by driverId and year, calculate mean pit stop duration for each driver/year pair
driver_pit_stops_summary = pit_stops_w_constructors.groupby(['driverId', 'year'], as_index=False).agg(avg_pit_stop_duration = ('duration','mean'))
# Group pit_stops by constructorId and year, calculate mean pit stop duration for each constructor/year pair
constructor_pit_stops_summary = pit_stops_w_constructors.groupby(['constructorId', 'year'], as_index=False).agg(avg_pit_stop_duration = ('duration','mean'))

In [5]:
# Convert milliseconds column to seconds for lap_times df
lap_times['duration'] = lap_times['milliseconds'] * 10**-3

# Standardize lap times for each race
avg_lap_times = lap_times.groupby('raceId').agg(avg_lap = ('duration', 'mean')) # calculate mean lap time for each raceId
lap_times_w_avg = pd.merge(lap_times, avg_lap_times, on='raceId')   # merge with original lap_times df by raceId so there is a mean lap time col
lap_times_w_avg['duration_standard'] = lap_times_w_avg['duration'] / lap_times_w_avg['avg_lap'] # calculate standardized lap times by dividing by mean lap time

# Add years column to lap_times df by merging with races df on raceId
lap_times_w_years = pd.merge(lap_times_w_avg, races[['raceId', 'year']], on='raceId')
# Add constructorId to lap_times df by merging with results on raceId and driverId
lap_times_w_constructors = pd.merge(lap_times_w_years, results[['raceId', 'driverId', 'constructorId']], on=['raceId', 'driverId'])

# Group lap_times by driverId and year, calculate standard deviation of standardized lap times for each driver/year pair
driver_lap_times_summary = lap_times_w_constructors.groupby(['driverId', 'year'], as_index=False).agg(stdev_lap_time = ('duration_standard', 'std'))
# Group lap_times by constructorId and year, calculate standard deviation of standardized lap times for each constructor/year pair
constructor_lap_times_summary = lap_times_w_constructors.groupby(['constructorId', 'year'], as_index=False).agg(stdev_lap_time = ('duration_standard', 'std'))

In [6]:
# Convert fastestLap column in results df to numeric, replacing all '\N' with np.nan
results['fastestLap'] = pd.to_numeric(results['fastestLap'].replace(r'\N', np.nan))
# Create fastest_lap_pos column, which shows (as a decimal) how far through the race each driver had their fastest lap
results['fastest_lap_pos'] = results['fastestLap'] / results['laps']

# Add years column to results df by merging with races df on raceId
results_w_years = pd.merge(results, races[['raceId', 'year']], on='raceId')

# Group results by driverId and year, calculate mean fastest lap position for each driver/year pair
driver_fastest_lap_summary = results_w_years.groupby(['driverId', 'year'], as_index=False).agg(fastest_lap = ('fastest_lap_pos', 'mean'))
# Group results by constructorId and year, calculate mean fastest lap position for each constructor/year pair
constructor_fastest_lap_summary = results_w_years.groupby(['constructorId', 'year'], as_index=False).agg(fastest_lap = ('fastest_lap_pos', 'mean'))

In [7]:
# Merge all variables calculated in cells above by driver/year pair into master drivers dataframe for regression analysis
drivers_df = pd.merge(drivers[['driverId', 'forename', 'surname', 'nationality']], driver_standings_summary, on='driverId')
drivers_df = pd.merge(drivers_df, driver_pit_stops_summary, on=['driverId', 'year'])
drivers_df = pd.merge(drivers_df, driver_lap_times_summary, on=['driverId', 'year'])
drivers_df = pd.merge(drivers_df, driver_fastest_lap_summary, on=['driverId', 'year'])

In [8]:
# Merge all variables calculated in cells above by constructor/year pair into master constructors dataframe for regression analysis
constructors_df = pd.merge(constructors[['constructorId', 'nationality']], constructor_standings_summary, on='constructorId')
constructors_df = pd.merge(constructors_df, constructor_pit_stops_summary, on=['constructorId', 'year'])
constructors_df = pd.merge(constructors_df, constructor_lap_times_summary, on=['constructorId', 'year'])
constructors_df = pd.merge(constructors_df, constructor_fastest_lap_summary, on=['constructorId', 'year'])

In [9]:
# Create a dictionary assigning all nationalities in constructor or driver nationality columns to regions
region_dict = {'Asia':['Indian', 'Indonesian', 'Thai', 'Malaysian', 'Japanese', 'Chinese'], 
               'Americas':['American', 'Canadian', 'Mexican', 'Brazilian', 'Venezuelan'],
               'Europe':['German', 'Spanish', 'Finnish', 'Polish', 'Italian', 'Swiss', 'French', 'Russian', 'Belgian', 'Dutch', 'Danish', 'Swedish', 'Monegasque'],
               'UK':['British'],
               'Oceania':['Australian', 'New Zealander']}
# Reverse the dictionary, so each nationality is a key with its region as the value
nat_to_region = {
    nationality: region
    for region, nat_list in region_dict.items()
    for nationality in nat_list
}

# Use reversed dictionary to create region var in master drivers and constructors dataframes
drivers_df["region"] = drivers_df["nationality"].map(nat_to_region)
constructors_df["region"] = constructors_df["nationality"].map(nat_to_region)

In [ ]:
# check whether constructor id or driver alone is a better predictor of points
# using C() so that the model treats id as categorical not numeric
constructors_model1 = smf.ols('mean_points ~ C(constructorId)', data=constructors_df).fit()

drivers_model1 = smf.ols("mean_points ~ C(driverId)", data=drivers_df).fit() 

print("Driver R²:", drivers_model1.rsquared)
print("Constructor R²:", constructors_model1.rsquared)

# 65% of variation in points is explained by driver
# 73.5% of variation in points is explained by constructor 
# constructor is a better predictor of performance ig

Driver R²: 0.650108175322553
Constructor R²: 0.7353715262388892


In [ ]:
# REMINDERS - ONLY X CHARACTERS PER LINE
#  okay so now im gonna attempt to split ts into test and training data
np.random.seed(151)

# divide data into 80% training and 20% testing

constructors_df['rand'] = np.random.rand(len(constructors_df))
constructors_training = constructors_df.query("rand < 0.8")
constructors_test = constructors_df.query("rand >= 0.8")

drivers_df['rand'] = np.random.rand(len(drivers_df))
drivers_training = drivers_df.query("rand < 0.8")
drivers_test = drivers_df.query("rand >= 0.8")

In [ ]:
# now lets do some linear regressions
# can't use constructor id and driver id as they don't exist in test and training and they're categorical

# using C() so that the model treats id as categorical not numeric
constructors_model2 = smf.ols('mean_points ~ nationality+avg_pit_stop_duration+stdev_lap_time', data=constructors_training).fit()

drivers_model2 = smf.ols('mean_points ~ nationality+avg_pit_stop_duration+stdev_lap_time', data=drivers_training).fit() 

# now calculate predicted values
drivers_pred2 = drivers_model2.predict(drivers_test)

constructors_pred2 = constructors_model2.predict(constructors_test)

# now check predictive accuracy by comparing mean squared error
drivers_mse2 = ((drivers_test['mean_points'] - drivers_pred2)**2).mean()

constructors_mse2 = ((constructors_test['mean_points'] - constructors_pred2)**2).mean()

# compare
print('Driver MSE:', drivers_mse2)
print('Costructor MSE:', constructors_mse2)

# the model works better on predicting driver performance over constructor performance

Driver MSE: 1834.4165276718745
Costructor MSE: 4628.376142270534
